In [0]:
%sql
-- PRODUCTION-CORRECT: mask SSN unless the user is in the full-PHI group
CREATE OR REPLACE FUNCTION meridian_dev.silver.mask_ssn(ssn STRING)
RETURN CASE
  WHEN is_account_group_member('meridian_phi_full') THEN ssn
  ELSE 'XXX-XX-' || right(ssn, 4)     -- show only last 4, HIPAA-style
END;

In [0]:
%sql
-- FREE-EDITION RUNNABLE: same mechanism, keyed on current user instead of group
CREATE OR REPLACE FUNCTION meridian_dev.silver.mask_ssn(ssn STRING)
RETURN CASE
  WHEN current_user() = 'ADMIN_PLACEHOLDER' THEN ssn   -- stand-in for "full-PHI role"
  ELSE 'XXX-XX-' || right(ssn, 4)
END;

In [0]:
%sql
ALTER TABLE meridian_dev.bronze.patients
  ALTER COLUMN SSN SET MASK meridian_dev.silver.mask_ssn;

In [0]:
%sql
SELECT Id, FIRST, LAST, SSN FROM meridian_dev.bronze.patients LIMIT 5;